# Choosing a Training Method — a Use-Case Lab

A **conceptual companion** to Parts 1–4. No model training here — the goal is judgment: given a real situation, *which* adaptation method do you reach for, and why? You've now built the mechanics of continued pretraining, full fine-tuning, a frozen-encoder probe, FinBERT, and QLoRA. This notebook is about knowing **when** to use each.

- **Part A — Triage deck:** 8 scenarios. Commit to a method + a one-line justification, then check the answer-key notebook.
- **Part B — Predict-the-ranking:** predict how your own methods rank on accuracy *and* cost, then fill the table and reconcile.

*(This is the `PRACTICE` file if the title says so — write your answers here, then open `Choosing_a_Training_Method.ipynb` to compare.)*

## The framework: what are you adapting, and what does it cost?

Every method changes a **different amount** of the model. From cheapest to most invasive:

| Method | What changes | Labeled data | Compute | Your example |
|---|---|---|---|---|
| **Prompting / zero-shot** | nothing | none | inference only | FinBERT's own head (P3 §8a) |
| **Freeze backbone, train head** | a tiny linear layer | very little | cheap | P3 §8b probe |
| **Full fine-tune** | every weight | moderate–lots | expensive | Parts 2 & 3 |
| **LoRA / QLoRA** | small adapters (~1%) | moderate | cheap-ish, fits big models | Part 4 |
| **Continued pretraining** | the backbone's *domain sense*, from **unlabeled** text | none (unlabeled) | expensive, but reusable | Part 1 (FinBERT productizes it) |
| **Pretrain from scratch** | everything, from zero | none (unlabeled, huge) | enormous | (almost never) |

### The five decision drivers
Whenever you choose, weigh these five:
1. **Labeled data** — how many *labeled* examples? (Little ⇒ freeze/prompt; lots ⇒ full fine-tune.)
2. **Domain distance** — how far is your text from the base model's pretraining? (Far ⇒ pretraining / a domain model helps.)
3. **Compute & memory** — what hardware/budget? (Tight ⇒ freeze or QLoRA, not full FT of a big model.)
4. **Number of downstream tasks** — one, or many on the same domain? (Many ⇒ pretraining / shared frozen backbone amortizes.)
5. **Serving constraints** — latency, cost, maintainability in production? (Strict ⇒ small/shared models.)

**The meta-rule:** pick the *least-effort* method that clears your accuracy bar; escalate only when it doesn't.

## Part A — Scenario triage  &larr; **your judgment**

For each scenario, fill in the `✍️ Your call` cell: the **method**, the **driver(s)** that tipped it, a one-sentence **why**, and **what would change your answer**. Don't aim for a single 'right' token — the reasoning is the point. Then compare with the answer key.

*(There isn't always one correct method; several scenarios are deliberately a judgment call between two.)*

### Scenario 1: FinBERT-style investment

A fintech has **5 billion words** of unlabeled analyst reports and filings, but only **~500 labeled** sentiment sentences. They plan to build **several** classifiers on this domain (sentiment, risk-flagging, topic). Generic LLMs handle finance jargon poorly. A multi-GPU budget is available.

#### ✍️ Your call — Scenario 1
- **Method:** Continued pretraining
- **Driver(s) that tipped it:** Large volume of unlabeled data from reports and filings
- **Why (1 sentence):** Our data is unlabeled, so we can't do fine tuning. We can take a model and use our data to make it more domain aware of our data.
- **What would change your answer:** The amount of labeled data - if we had labels for the sentences of the analyst report and filings we could do a full fine-tuning instead of pretraining.

### Scenario 2: Tiny labeled set, ships next week

A startup has **200 labeled** examples for **one** classification task, a **strong general** base model, and a **single modest GPU**. Accuracy needs to be 'decent,' and it ships in a week.

#### ✍️ Your call — Scenario 2
- **Method:** QLoRA, Train head
- **Driver(s) that tipped it:** GPU limitation
- **Why (1 sentence):** Quantization will speed up the training process and allow the startup to make the most of their strong model with limited resources.
- **What would change your answer:**  To make the most of a strong model - more data with a better GPU could let them do a full fine tuning instead of just training a head

### Scenario 3: Plenty of data, close domain

**80,000 labeled** product reviews, **one** sentiment task. The general base model is **already strong** on consumer text. GPUs are available, and you want the **best** achievable accuracy.

#### ✍️ Your call — Scenario 3
- **Method:** Full fine tuning
- **Driver(s) that tipped it:** Best achievable accuracy goal - data and GPU are not limitations
- **Why (1 sentence):** Many examples of labeled data for one sentiment tasks is enough to do a full fine-tuning rather than just trainign a NN head
- **What would change your answer:** Availability of compute resources - with limited compute they may only have resources for training the head

### Scenario 4: Big model, small GPU

You want the reasoning quality of a **13B** LLM for a classification/extraction task, you have **~10,000 labeled** examples, but only a **single 16 GB GPU**.

#### ✍️ Your call — Scenario 4
- **Method:** QLoRA
- **Driver(s) that tipped it:** limited GPU memory
- **Why (1 sentence):** The model may not fit on the GPU at all without quantization.
- **What would change your answer:** GPU size - 10k labeled examples is enough to fine time a model but the compute is the limitation

### Scenario 5: Need it today, no labels

A PM asks for a rough sentiment tagger **by tomorrow**. There's **no labeled data**, no training pipeline, but you have API access to a **capable general LLM**.

#### ✍️ Your call — Scenario 5
- **Method:** Prompting
- **Driver(s) that tipped it:** No training data, and no development time
- **Why (1 sentence):** The short timeline means we have to use a commercial model.
- **What would change your answer:** In this scenario multiple factors point toward prompting a commercial model - < 24 hr turn around, no labeled data, no training pipeline. You'd need to fix  those issues to use any other technique.

### Scenario 6: Many tasks, one backbone, low latency

A platform must run **12 different** text classifiers (sentiment, topic, toxicity, language, …) **cheaply** and with **low latency**. Maintaining 12 separately fine-tuned full models is too costly.

#### ✍️ Your call — Scenario 6
- **Method:** Prompting, QLoRA
- **Driver(s) that tipped it:** 1 model to do 12 different tasks requires 12 different prompts
- **Why (1 sentence):** Let's assume we have a domain pretrained model but we don't want to create 12 different versions for individual tasks. We can use QLoRA to speed up the latency and reduce compute resources
- **What would change your answer:** Resources are the main limitation. We are making a big tradeoff by limiting our maintenance to 1 model instead of 12 fine-tuned models.

### Scenario 7: Tempted to pretrain — but should you?

You have a **large unlabeled finance corpus** and **one** upcoming sentiment task. You're tempted to continue-pretrain GPT-2 for **weeks**. The deadline is tight — and **FinBERT already exists**.

#### ✍️ Your call — Scenario 7
- **Method:** Prompting
- **Driver(s) that tipped it:** pre-training is too risky with a tight deadline
- **Why (1 sentence):** Weeks of training is a huge cost which introduces risk to a tight deadline scenario. Training/pre-training is more appropriate as a long term investment, not a quick fix.
- **What would change your answer:** Timeline

### Scenario 8: No suitable base model exists

A genomics lab has **billions of DNA-sequence tokens**, a **unique vocabulary** unlike any pretrained model, a **large compute grant**, and **many** planned downstream predictors.

#### ✍️ Your call — Scenario 8
- **Method:** Train from scratch
- **Driver(s) that tipped it:** unique vocabulary unlike any pre-trained model
- **Why (1 sentence):** Pre-trained models will not be able to understand the DNA sequence tokens that use a different vocabulary
- **What would change your answer:** The vocabulary/data domain. If we can get a head start with a pre-trained model that would save a lot of time and money

### Part A wrap-up

Notice the recurring tells:
- **Few labels** → freeze / prompt. **Many labels** → full fine-tune.
- **Far domain + unlabeled text + many tasks** → pretraining (or grab a domain model). **One task or close domain** → don't.
- **Big model, small GPU** → QLoRA. **Many tasks, tight serving** → shared frozen backbone.
- **No time / no labels** → prompt, then distill later.

Almost every real decision is just these five drivers pulling against each other.

## Part B — Predict the ranking  &larr; **your judgment, then your data**

You've now run (or will run) the same Financial PhraseBank task with up to seven methods. Before looking at the numbers, **commit to two predictions**. Then fill the table and see where reality surprised you.

#### ✍️ Before you peek — commit to two rankings
Rank these seven on **test accuracy, best → worst** (just reorder them):
GPT-2 base · GPT-2 EDGAR · BERT-base FT · FinBERT FT · FinBERT zero-shot · BERT frozen probe · Mistral QLoRA
1. FinBERT FT
2. FinBERT zero-shot
3. GPT-2 EDGAR
4. Mistral QLoRA
5. BERT-base FT
6. BERT frozen probe
7. GPT-2 base

Now rank them **cheapest → most expensive to train**:
1. GPT-2 base
2. GPT-2 EDGAR
3. Mistral QLoRA
4. FinBERT zero-shot
5. BERT frozen probe
6. BERT-base FT
7. FinBERT FT

**One sentence — what's the biggest surprise you expect?** GPT-2 EDGAR with domain knowledge outperforming bigger models without domain training

### The results table
Fill `test_acc` from **your** runs (leave `None` for parts you haven't run yet), and assign each method a rough `train_cost` on a **0–5** ordinal (0 = no training, 5 = most expensive). The cell sorts it two ways so the ranking falls out.

In [1]:
import pandas as pd

# Fill test_acc from YOUR runs (None until you've run that part).
# Assign each method a rough train_cost on a 0-5 ordinal (0 = no training, 5 = most expensive) -- YOUR call.
rows = [
    # method,                              part,     test_acc, train_cost, notes
    ('GPT-2 fine-tuned (base)',            'P2',      None,    None, ''),
    ('GPT-2 fine-tuned (EDGAR-pretrn)',    'P2',      None,    None, ''),
    ('BERT-base [CLS] fine-tuned',         'P3',      None,    None, ''),
    ('FinBERT [CLS] fine-tuned',           'P3',      None,    None, ''),
    ('FinBERT zero-shot (own head)',       'P3 8a',   None,    None, ''),
    ('BERT-base frozen probe',             'P3 8b',   None,    None, ''),
    ('Mistral-7B QLoRA',                   'P4',      None,    None, ''),
]
df = pd.DataFrame(rows, columns=['method', 'part', 'test_acc', 'train_cost', 'notes'])

print('--- by ACCURACY (best first) ---')
print(df.sort_values('test_acc', ascending=False, na_position='last').to_string(index=False))
print('\n--- by TRAINING COST (cheapest first) ---')
print(df.sort_values('train_cost', ascending=True, na_position='last').to_string(index=False))

--- by ACCURACY (best first) ---
                         method  part test_acc train_cost notes
        GPT-2 fine-tuned (base)    P2     None       None      
GPT-2 fine-tuned (EDGAR-pretrn)    P2     None       None      
     BERT-base [CLS] fine-tuned    P3     None       None      
       FinBERT [CLS] fine-tuned    P3     None       None      
   FinBERT zero-shot (own head) P3 8a     None       None      
         BERT-base frozen probe P3 8b     None       None      
               Mistral-7B QLoRA    P4     None       None      

--- by TRAINING COST (cheapest first) ---
                         method  part test_acc train_cost notes
        GPT-2 fine-tuned (base)    P2     None       None      
GPT-2 fine-tuned (EDGAR-pretrn)    P2     None       None      
     BERT-base [CLS] fine-tuned    P3     None       None      
       FinBERT [CLS] fine-tuned    P3     None       None      
   FinBERT zero-shot (own head) P3 8a     None       None      
         BERT-base frozen pr

### ✍️ Reconcile — your turn
Answer after you've filled the table:
- Where did the **accuracy** ranking differ from your prediction? _…_
- Where did the **cost** ranking differ? _…_
- How big is the gap between the **frozen probe** and the full fine-tune — and what does that gap *mean*? _…_
- Is the most expensive method (Mistral QLoRA) worth it **for this task**? Why / why not? _…_
- **Restate the meta-rule in your own words:** _…_